# Laboratorio 3: Trafilatura, Pandas & CSV

Hasta este punto de la cursada construimos los procesos de extracción manualmente mediante Beautiful Soup. Ahora incorporaremos inteligencia artificial heurística a nuestro proceso de scraping introduciendo **Trafilatura**. Esta moderna herramienta está diseñada exclusivamente para detectar y aislar el núcleo textual de artículos de noticias o blogs, ignorando componentes estructurales secundarios, publicidades, menús y pies de página.

Dado que no resulta profesional extraer información para imprimirla sin estructura en consola, este laboratorio cierra el circuito analítico generando un **DataFrame** de Pandas. De esta forma, el texto extraído se consolida en un corpus estructurado, preparado para exportarse o alimentar un modelo analítico subsecuente.

In [1]:
# Instalamos la herramienta
!pip install trafilatura -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import trafilatura

In [3]:
url = "https://cenital.com/epstein-y-la-decadencia-del-sistema-cientifico/"

Notá la capacidad heurística de Trafilatura. Con dos simples sentencias descarga la página completa y, mediante la función `.extract()`, depura la interfaz gráfica, retornando de forma directa el texto periodístico legible.

In [4]:
downloaded = trafilatura.fetch_url(url)

# Parametrizamos la extracción para obviar comentarios e imágenes explícitamente
text = trafilatura.extract(downloaded, include_comments=False, include_images=False)
print(text)

Epstein y la decadencia del sistema científico
La publicación de los archivos del multimillonario mostró su interés por infiltrarse en la élite académica y cómo lo hacía. Una falla estructural en el sistema que produce conocimiento.
El escándalo de Jeffrey Epstein suele apoyarse en escenografías de islas privadas, jets y abusos repugnantes. La trama del multimillonario pedófilo se escribe sola y no hace más que confirmar las peores sospechas acerca de las élites.
Pero Epstein no solo se rodeaba de menores de edad. También procuraba la compañía de científicos, si no altos y esbeltos, con abundantes credenciales académicas capaces de sonrojar a cualquiera. “Solo tengo dos intereses”, Epstein le dijo una vez a un amigo: “ciencia y mujeres” (la cita exacta es “science and pussy”, de una sutileza poética difícil de traducir).
Esto es demostrable: tras la publicación de millones de documentos, imágenes, videos y correos electrónicos que detallan sus actividades, conocidos como los “archivos 

### Escaneando a Gran Escala con Pandas 
A partir de aquí elevamos la complejidad. Resulta imperativo encapsular las tareas de alto riesgo de red dentro de bloques `try...except`. Mediante esta práctica, si una URL en una lista extensa se encuentra inaccesible, el proceso absorbe internamente la excepción y el script continúa procesando los siguientes enlaces sin interrupciones críticas.

In [7]:
import pandas as pd
import trafilatura

def obtener_texto_articulo(url):
    # Encapsulamos la lógica sensible frente a potenciales caídas de internet
    try:
        descargado = trafilatura.fetch_url(url)
        if descargado:
            texto = trafilatura.extract(descargado, include_comments=False, include_images=False)
            if texto:
                return texto
            else:
                return "No se pudo extraer texto de la URL."
        else:
            return "No se pudo obtener la URL."
    except Exception as e:
        return f"Ocurrió un error: {e}"


In [8]:
# Lista iterable para procesar flujos en lote (Batching)
urls = [
    "https://cenital.com/epstein-y-la-decadencia-del-sistema-cientifico/",
    "https://cenital.com/el-cerebro-tal-vez-no-sea-una-computadora/",
    "https://cenital.com/la-fragilidad-de-internet/",
]

corpus_data = []

# Repetimos nuestro algoritmo por cada enlace provisto
for url in urls:
    print(f"Procesando el enlace: {url}")
    texto_extraido = obtener_texto_articulo(url)
    
    # Empaquetamos resultados en un diccionario y lo adjuntamos al corpus temporal
    corpus_data.append({'URL': url, 'Texto Principal': texto_extraido})

# Inicializamos nuestra estructura de datos permanente con Pandas
df_corpus = pd.DataFrame(corpus_data)

display(df_corpus.head())

Procesando el enlace: https://cenital.com/epstein-y-la-decadencia-del-sistema-cientifico/
Procesando el enlace: https://cenital.com/el-cerebro-tal-vez-no-sea-una-computadora/
Procesando el enlace: https://cenital.com/la-fragilidad-de-internet/


,URL,Texto Principal
0,https://cenital.com/epstein-y-la-decadencia-de...,Epstein y la decadencia del sistema científico...
1,https://cenital.com/el-cerebro-tal-vez-no-sea-...,El cerebro tal vez no sea una computadora\nEl ...
2,https://cenital.com/la-fragilidad-de-internet/,La fragilidad de Internet\nLa caída de un serv...


Para no malgastar capacidad de procesamiento y tiempo de red, es esencial persistir el trabajo realizado (Persistencia de Datos). Mediante Pandas, una acción compleja como exportar nuestras variables a un archivo estático separado por comas requiere únicamente la siguiente instrucción:

In [9]:
# Guardar a CSV ignorando que te exporte el índice numérico que puso
df_corpus.to_csv('gran_corpus_noticias.csv', index=False)
print("¡Datos guardados exitosamente!")

¡Datos guardados exitosamente!


Por último, observaremos cómo recuperar programáticamente las piezas textuales alojadas dentro del DataFrame de Pandas.

In [10]:
# Recorré e imprimí filas de noticias usando iterrows
for index, row in df_corpus.iterrows():
    print(f"--- Texto del artículo de la URL: {row['URL']} ---")
    print(row['Texto Principal'])
    print("-" * 30) 

--- Texto del artículo de la URL: https://cenital.com/epstein-y-la-decadencia-del-sistema-cientifico/ ---
Epstein y la decadencia del sistema científico
La publicación de los archivos del multimillonario mostró su interés por infiltrarse en la élite académica y cómo lo hacía. Una falla estructural en el sistema que produce conocimiento.
El escándalo de Jeffrey Epstein suele apoyarse en escenografías de islas privadas, jets y abusos repugnantes. La trama del multimillonario pedófilo se escribe sola y no hace más que confirmar las peores sospechas acerca de las élites.
Pero Epstein no solo se rodeaba de menores de edad. También procuraba la compañía de científicos, si no altos y esbeltos, con abundantes credenciales académicas capaces de sonrojar a cualquiera. “Solo tengo dos intereses”, Epstein le dijo una vez a un amigo: “ciencia y mujeres” (la cita exacta es “science and pussy”, de una sutileza poética difícil de traducir).
Esto es demostrable: tras la publicación de millones de docum

In [11]:
# Apuntamos derechamente al artículo cero de la colección y su columna Texto
print(df_corpus.loc[0, 'Texto Principal'])

Epstein y la decadencia del sistema científico
La publicación de los archivos del multimillonario mostró su interés por infiltrarse en la élite académica y cómo lo hacía. Una falla estructural en el sistema que produce conocimiento.
El escándalo de Jeffrey Epstein suele apoyarse en escenografías de islas privadas, jets y abusos repugnantes. La trama del multimillonario pedófilo se escribe sola y no hace más que confirmar las peores sospechas acerca de las élites.
Pero Epstein no solo se rodeaba de menores de edad. También procuraba la compañía de científicos, si no altos y esbeltos, con abundantes credenciales académicas capaces de sonrojar a cualquiera. “Solo tengo dos intereses”, Epstein le dijo una vez a un amigo: “ciencia y mujeres” (la cita exacta es “science and pussy”, de una sutileza poética difícil de traducir).
Esto es demostrable: tras la publicación de millones de documentos, imágenes, videos y correos electrónicos que detallan sus actividades, conocidos como los “archivos 

---
# Consolidación y Repaso

## Glosario Acotado
*   **Corpus:** Colección masiva y estructurada de textos, indispensable en los procesos de lingüística computacional y crucial para el entrenamiento o ajuste (fine-tuning) de modelos de Inteligencia Artificial.
*   **Trafilatura:** Librería específica enfocada en la extracción de artículos o 'article scraping'. Analiza el DOM estructural y depura la interfaz, menús y elementos publicitarios, extrayendo con alta certeza algorítmica el bloque editorial original.
*   **DataFrame (Pandas):** Estructura de datos matricial bidimensional provista por la biblioteca Pandas. Representa el estándar universal en Python para la tabulación, exploración y limpieza de variables analíticas.
*   **Persistencia de datos:** Etapa del proceso informático donde la información alojada en la memoria volátil de ejecución (RAM) se transcribe permanentemente hacia una unidad de almacenamiento no volátil (como un archivo CSV).

## Preguntas de Autoevaluación

**1. ¿Qué ventaja sustancial plantea apoyarnos en Trafilatura en lugar del análisis manual con Beautiful Soup (BS4)?**
Los modelos heurísticos de Trafilatura están pre-entrenados para diferenciar estadísticamente el contenido semántico principal del marcado periférico (como módulos de navegación o banners de publicidad). Previamente, con BS4, dependíamos de ubicar e identificar manualmente y caso por caso cada `<div>` estructural que envolviera nuestra información para no capturar texto basura.

**2. ¿Por qué es necesario culminar el proceso estructurando un DataFrame de Pandas al finalizar la extracción?**
Visualizar datos mediante impresiones de consola (prints) es útil estrictamente para pruebas diagnósticas iniciales. Un ingeniero en datos y desarrollo de web scraping debe proporcionar salidas estructuradas y formalizadas, ya que la meta de estos programas frecuentemente es delegar ese corpus consolidado a terceros (como analistas matemáticos, integradores o sistemas BI). Una variable plana o un diccionario desestructurado no permite el análisis matricial estadístico profundo requerido a nivel profesional.